# Hurricane Melissa: Forest NDVI >10% Damage Distance Analysis

This notebook replicates the **continuous distance-to-track concentration** analysis for **forest-equivalent pixels** (not mangroves), using:
- NDVI before/after rasters (EPSG:3448)
- 2013 landcover with `forest_flood_equivalent_classes` and `mixed_land_use_fractions`
- Melissa track line (NOAA best track)

Damage definition:
- Relative NDVI change `(after - before) / before < -0.10`
- Eligible pixels only where `NDVI before >= 0.20`


In [ ]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from shapely.ops import unary_union


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')

ROOT = find_project_root(Path.cwd())

# NDVI + landcover + class definitions
ndvi_before_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif'
ndvi_after_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif'
landcover_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_landcover.gpkg'
robyn_defs_path = ROOT / 'dphil_papers/robyns_libraries/Robyn_catchment_analysis.py'

# Hurricane Melissa track
track_dir = ROOT / 'dphil_papers/dphil_paper_3/inputs/hurricane_melissa_track_noaa/al132025_best_track'
line_path = track_dir / 'AL132025_lin.shp'
windswath_path = track_dir / 'AL132025_windswath.shp'
jamaica_boundary_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

REL_DAMAGE_THRESHOLD = -0.10
REL_BASELINE_MIN = 0.20
TRACK_AOI_BUFFER_KM = 200

# Reuse/caching of geospatial eligible forest pixels for distance analysis
forest_pairs_csv = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images/ndvi_forests_paired_pixels_with_weights.csv'
forest_eligible_cache = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images/ndvi_forests_eligible_pixels_epsg3448.parquet'

# Reuse/caching of full-grid forest relative-change class raster (-1/0/1)
forest_rel_class_raster = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images/ndvi_forests_relative_substantial_change_gt10_classes_epsg3448.tif'

for p in [ndvi_before_path, ndvi_after_path, landcover_path, robyn_defs_path, line_path, windswath_path, jamaica_boundary_path, forest_pairs_csv]:
    print(p.name, 'exists ->', p.exists())
print('Eligible forest cache path:', forest_eligible_cache)
print('Forest class raster path:', forest_rel_class_raster)


In [ ]:
# Parse forest class definitions from Robyn script
source = robyn_defs_path.read_text()
tree = ast.parse(source)

forest_flood_equivalent_classes = None
mixed_land_use_fractions_primary = None

for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name):
                if target.id == 'forest_flood_equivalent_classes' and forest_flood_equivalent_classes is None:
                    forest_flood_equivalent_classes = set(ast.literal_eval(node.value))
                if target.id == 'mixed_land_use_fractions':
                    candidate = ast.literal_eval(node.value)
                    if (
                        isinstance(candidate, dict)
                        and any(isinstance(v, dict) and 'forest_flood_equivalent_classes' in v for v in candidate.values())
                    ):
                        mixed_land_use_fractions_primary = candidate

if forest_flood_equivalent_classes is None or mixed_land_use_fractions_primary is None:
    raise ValueError('Could not parse forest classes/fractions from Robyn_catchment_analysis.py')

print('Forest flood-equivalent classes:', len(forest_flood_equivalent_classes))
print('Mixed classes with fractions:', len(mixed_land_use_fractions_primary))


In [ ]:
def forest_fraction_for_class(class_name: str) -> float:
    class_name = str(class_name)
    if class_name in mixed_land_use_fractions_primary:
        return float(mixed_land_use_fractions_primary[class_name].get('forest_flood_equivalent_classes', 0.0))
    if class_name in forest_flood_equivalent_classes:
        return 1.0
    return 0.0

if forest_eligible_cache.exists():
    eligible_pixels = gpd.read_parquet(forest_eligible_cache)
    if eligible_pixels.crs is None:
        eligible_pixels = eligible_pixels.set_crs(3448, allow_override=True)
    else:
        eligible_pixels = eligible_pixels.to_crs(3448)
    print('Loaded cached eligible forest pixels:', forest_eligible_cache)
else:
    print('No geospatial eligible cache found; rebuilding once from NDVI + landcover and saving cache...')

    with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after_path) as src_a:
        if src_b.crs != src_a.crs or src_b.transform != src_a.transform or src_b.shape != src_a.shape:
            raise ValueError('Before/after NDVI rasters are not on same grid.')

        ndvi_before = src_b.read(1)
        ndvi_after = src_a.read(1)
        transform = src_b.transform
        crs = src_b.crs
        shape = src_b.shape

        landcover = gpd.read_file(landcover_path, columns=['Classify', 'geometry'])
        if landcover.crs != crs:
            landcover = landcover.to_crs(crs)

        landcover = landcover[landcover.geometry.notnull() & ~landcover.geometry.is_empty].copy()
        landcover['forest_fraction'] = landcover['Classify'].astype(str).map(forest_fraction_for_class)
        forest_landcover = landcover[landcover['forest_fraction'] > 0].copy()

        shapes = ((geom, float(frac)) for geom, frac in zip(forest_landcover.geometry, forest_landcover['forest_fraction']))
        forest_fraction_grid = rasterize(
            shapes,
            out_shape=shape,
            transform=transform,
            fill=0.0,
            dtype='float32'
        )

        valid_before = np.isfinite(ndvi_before) & (ndvi_before >= -1.0) & (ndvi_before <= 1.0)
        valid_after = np.isfinite(ndvi_after) & (ndvi_after >= -1.0) & (ndvi_after <= 1.0)

        if src_b.nodata is not None and np.isfinite(src_b.nodata):
            valid_before &= ndvi_before != src_b.nodata
        if src_a.nodata is not None and np.isfinite(src_a.nodata):
            valid_after &= ndvi_after != src_a.nodata

    valid_forest = forest_fraction_grid > 0
    paired_forest = valid_before & valid_after & valid_forest
    eligible = paired_forest & (ndvi_before >= REL_BASELINE_MIN)

    delta = ndvi_after - ndvi_before
    rel_change = np.full(ndvi_before.shape, np.nan, dtype='float32')
    rel_change[eligible] = delta[eligible] / ndvi_before[eligible]

    rows, cols = np.where(eligible)
    xs, ys = rasterio.transform.xy(transform, rows, cols, offset='center')

    eligible_pixels = gpd.GeoDataFrame(
        {
            'ndvi_before': ndvi_before[eligible],
            'ndvi_after': ndvi_after[eligible],
            'ndvi_delta': delta[eligible],
            'rel_change': rel_change[eligible],
            'forest_fraction': forest_fraction_grid[eligible],
            'damaged_gt10pct': rel_change[eligible] < REL_DAMAGE_THRESHOLD,
        },
        geometry=gpd.points_from_xy(xs, ys),
        crs=3448,
    )

    forest_eligible_cache.parent.mkdir(parents=True, exist_ok=True)
    eligible_pixels.to_parquet(forest_eligible_cache, index=False)
    print('Saved geospatial eligible forest cache:', forest_eligible_cache)

# Ensure required columns exist when loading cache from prior run
if 'damaged_gt10pct' not in eligible_pixels.columns:
    eligible_pixels['damaged_gt10pct'] = eligible_pixels['rel_change'] < REL_DAMAGE_THRESHOLD

print('Eligible forest pixels:', f'{len(eligible_pixels):,}')
print('Damaged forest pixels (>10% decline):', f"{int(eligible_pixels['damaged_gt10pct'].sum()):,}")


In [ ]:
def coerce_track_to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        gdf = gdf.set_crs(4326, allow_override=True)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
    return gdf

track_line_full = coerce_track_to_wgs84(gpd.read_file(line_path)).to_crs(3448)
track_windswath_full = coerce_track_to_wgs84(gpd.read_file(windswath_path)).to_crs(3448)
jamaica = gpd.read_file(jamaica_boundary_path).to_crs(3448)
track_aoi_geom = unary_union(jamaica.geometry).buffer(TRACK_AOI_BUFFER_KM * 1000)
track_aoi = gpd.GeoDataFrame(geometry=[track_aoi_geom], crs=3448)

track_line = gpd.overlay(track_line_full, track_aoi, how='intersection')
track_windswath = gpd.overlay(track_windswath_full, track_aoi, how='intersection')

if len(track_line) == 0:
    track_line = track_line_full.copy()
if len(track_windswath) == 0:
    track_windswath = track_windswath_full.copy()

line_union = unary_union(track_line.geometry.tolist())
print('Track line segments used:', len(track_line))
print('Windswath polygons used:', len(track_windswath))


## Notes
- This notebook is separate from your mangrove notebook, so existing files are untouched.
- Forest classes are pulled directly from `Robyn_catchment_analysis.py` (`forest_flood_equivalent_classes` and `mixed_land_use_fractions`).
- Distance analysis is continuous; peak and concentration intervals are data-driven.


## Added at End: Forest NDVI Change Map with Melissa Track

Map of eligible forest pixels using the same `>10%` relative NDVI-change classes:
- Red: decrease >10%
- Green: increase >10%
- Grey: other (not substantial)


In [ ]:
# Forest relative substantial-change map from geospatial output input + hurricane overlays
# Input-first: load saved class raster if it exists; build once if missing.

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, BoundaryNorm

REL_THRESHOLD = abs(float(REL_DAMAGE_THRESHOLD))  # 0.10
JAMAICA_MAP_BUFFER_KM = 60

if forest_rel_class_raster.exists():
    with rasterio.open(forest_rel_class_raster) as src:
        cls_rel = src.read(1).astype('float32')
        cls_rel = np.where(np.isclose(cls_rel, src.nodata) if src.nodata is not None else False, np.nan, cls_rel)
        bounds = src.bounds
        print('Loaded class raster input:', forest_rel_class_raster)
else:
    print('Class raster not found; building once, then saving for reuse:', forest_rel_class_raster)
    with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after_path) as src_a:
        if src_b.crs != src_a.crs or src_b.transform != src_a.transform or src_b.shape != src_a.shape:
            raise ValueError('Before/after NDVI rasters are not on same grid.')

        ndvi_before = src_b.read(1)
        ndvi_after = src_a.read(1)
        transform = src_b.transform
        shape = src_b.shape
        bounds = src_b.bounds
        crs = src_b.crs

        landcover = gpd.read_file(landcover_path, columns=['Classify', 'geometry'])
        if landcover.crs != crs:
            landcover = landcover.to_crs(crs)

        landcover = landcover[landcover.geometry.notnull() & ~landcover.geometry.is_empty].copy()
        landcover['forest_fraction'] = landcover['Classify'].astype(str).map(forest_fraction_for_class)
        forest_landcover = landcover[landcover['forest_fraction'] > 0].copy()

        shapes = ((geom, float(frac)) for geom, frac in zip(forest_landcover.geometry, forest_landcover['forest_fraction']))
        forest_fraction_grid = rasterize(
            shapes,
            out_shape=shape,
            transform=transform,
            fill=0.0,
            dtype='float32'
        )

        valid_before = np.isfinite(ndvi_before) & (ndvi_before >= -1.0) & (ndvi_before <= 1.0)
        valid_after = np.isfinite(ndvi_after) & (ndvi_after >= -1.0) & (ndvi_after <= 1.0)

        if src_b.nodata is not None and np.isfinite(src_b.nodata):
            valid_before &= ndvi_before != src_b.nodata
        if src_a.nodata is not None and np.isfinite(src_a.nodata):
            valid_after &= ndvi_after != src_a.nodata

    valid_forest = forest_fraction_grid > 0
    valid_paired_forest = valid_before & valid_after & valid_forest

    eligible_rel_full = valid_paired_forest & (ndvi_before >= REL_BASELINE_MIN)
    rel_change_full = np.full(ndvi_before.shape, np.nan, dtype='float32')
    rel_change_full[eligible_rel_full] = (ndvi_after[eligible_rel_full] - ndvi_before[eligible_rel_full]) / ndvi_before[eligible_rel_full]

    cls_rel = np.full(ndvi_before.shape, np.nan, dtype='float32')
    cls_rel[valid_paired_forest] = 0.0
    cls_rel[eligible_rel_full & (rel_change_full < -REL_THRESHOLD)] = -1.0
    cls_rel[eligible_rel_full & (rel_change_full > REL_THRESHOLD)] = 1.0

    forest_rel_class_raster.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        forest_rel_class_raster,
        'w',
        driver='GTiff',
        height=cls_rel.shape[0],
        width=cls_rel.shape[1],
        count=1,
        dtype='float32',
        crs=crs,
        transform=transform,
        nodata=-9999.0,
        compress='deflate'
    ) as dst:
        out = np.where(np.isnan(cls_rel), -9999.0, cls_rel).astype('float32')
        dst.write(out, 1)
    print('Saved class raster output:', forest_rel_class_raster)

n_dec = int(np.nansum(cls_rel == -1.0))
n_mid = int(np.nansum(cls_rel == 0.0))
n_inc = int(np.nansum(cls_rel == 1.0))
n_tot = n_dec + n_mid + n_inc

cmap = ListedColormap(['#d73027', '#d9d9d9', '#1a9850'])
cmap.set_bad(color='white', alpha=1.0)
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

fig, ax = plt.subplots(figsize=(12.5, 9.5), constrained_layout=True)
ax.imshow(
    cls_rel,
    cmap=cmap,
    norm=norm,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    origin='upper',
    interpolation='nearest',
)

# Overlay track and wind thresholds (same style family as mangrove map)
if len(track_windswath) > 0 and 'RADII' in track_windswath.columns:
    sw = track_windswath.copy()
    sw['RADII'] = pd.to_numeric(sw['RADII'], errors='coerce')
    wind_colors = {34: '#9ecae1', 50: '#4292c6', 64: '#08519c'}
    for thr in sorted(sw['RADII'].dropna().unique()):
        subset = sw[sw['RADII'] == thr]
        color = wind_colors.get(int(thr), '#6baed6')
        subset.boundary.plot(ax=ax, color=color, linewidth=1.2, alpha=0.9)

track_line.plot(ax=ax, color='black', linewidth=1.7, alpha=0.95)
jamaica.boundary.plot(ax=ax, color='black', linewidth=1.0, alpha=0.9)

jamaica_union = unary_union(jamaica.geometry)
map_extent = jamaica_union.buffer(JAMAICA_MAP_BUFFER_KM * 1000)
minx, miny, maxx, maxy = map_extent.bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

ax.set_title(
    f'Forest-equivalent Areas: Relative NDVI Substantial Change (>10%) with Melissa Track and Wind Thresholds\n'
    f'Baseline condition: NDVI before >= {REL_BASELINE_MIN:.2f}',
    fontsize=12,
)
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='NDVI decrease > 10%'),
    mpatches.Patch(facecolor='#d9d9d9', edgecolor='none', label='Not substantial / low baseline'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='NDVI increase > 10%'),
    Line2D([0], [0], color='black', lw=1.7, label='Melissa track line'),
]
if len(track_windswath) > 0 and 'RADII' in track_windswath.columns:
    for thr, clr in [(34, '#9ecae1'), (50, '#4292c6'), (64, '#08519c')]:
        if np.any(pd.to_numeric(track_windswath['RADII'], errors='coerce') == thr):
            legend_handles.append(Line2D([0], [0], color=clr, lw=1.6, label=f'{thr} kt wind threshold'))

ax.legend(
    handles=legend_handles,
    loc='lower left',
    frameon=True,
    framealpha=0.92,
    fontsize=8,
    handlelength=1.0,
    handleheight=0.9,
    borderpad=0.35,
    labelspacing=0.3,
)

ax.text(
    0.01,
    0.01,
    f'n={n_tot:,}  dec={100*n_dec/max(n_tot,1):.1f}%  other={100*n_mid/max(n_tot,1):.1f}%  inc={100*n_inc/max(n_tot,1):.1f}%',
    transform=ax.transAxes,
    fontsize=9,
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85, edgecolor='none'),
)

plt.show()


In [ ]:
# Continuous distance concentration stats (no random thresholds)
eligible_pixels['distance_to_track_km'] = eligible_pixels.geometry.distance(line_union) / 1000.0

dist_all = eligible_pixels['distance_to_track_km']
dist_dmg = eligible_pixels.loc[eligible_pixels['damaged_gt10pct'], 'distance_to_track_km']

if len(dist_dmg) == 0:
    raise ValueError('No damaged forest pixels found at >10% threshold.')

# Quantiles + cumulative concentration
q_levels = [0.10, 0.25, 0.50, 0.75, 0.90]
q_vals = np.quantile(dist_dmg, q_levels)
d_sorted = np.sort(dist_dmg.to_numpy())

def cum_distance(sorted_vals, p):
    i = int(np.ceil(p * len(sorted_vals))) - 1
    i = max(0, min(i, len(sorted_vals) - 1))
    return float(sorted_vals[i])

def hdi_interval(sorted_vals, p):
    n = len(sorted_vals)
    k = int(np.floor(p * n))
    if k < 1:
        return float(sorted_vals[0]), float(sorted_vals[-1])
    widths = sorted_vals[k:] - sorted_vals[:n-k]
    j = int(np.argmin(widths))
    return float(sorted_vals[j]), float(sorted_vals[j + k])

hdi50 = hdi_interval(d_sorted, 0.50)
hdi80 = hdi_interval(d_sorted, 0.80)

# Data-driven bins via Freedman-Diaconis for peak concentration
q1, q3 = np.quantile(dist_dmg, [0.25, 0.75])
iqr = q3 - q1
n = len(dist_dmg)
bw = 2 * iqr / (n ** (1/3)) if (iqr > 0 and n > 1) else max(0.5, float(np.std(dist_dmg)) / 10)
if (not np.isfinite(bw)) or (bw <= 0):
    bw = 0.5

mn, mx = float(np.min(dist_dmg)), float(np.max(dist_dmg))
nbins = int(np.ceil((mx - mn) / bw))
nbins = max(15, min(nbins, 200))
edges = np.linspace(mn, mx, nbins + 1)
centers = 0.5 * (edges[:-1] + edges[1:])

hist_dmg, _ = np.histogram(dist_dmg, bins=edges)
hist_all, _ = np.histogram(dist_all, bins=edges)
rate = np.divide(hist_dmg, hist_all, out=np.full_like(hist_dmg, np.nan, dtype=float), where=hist_all > 0)

kernel = np.array([1, 2, 3, 2, 1], dtype=float)
kernel /= kernel.sum()
sm_dmg = np.convolve(hist_dmg, kernel, mode='same')
sm_rate = np.convolve(np.nan_to_num(rate, nan=0.0), kernel, mode='same')

peak_count_idx = int(np.nanargmax(sm_dmg))
peak_rate_idx = int(np.nanargmax(sm_rate))

summary_distance = pd.DataFrame([
    {'metric': 'Eligible forest pixels', 'value': int(len(eligible_pixels))},
    {'metric': 'Damaged forest pixels (>10% decline)', 'value': int(eligible_pixels['damaged_gt10pct'].sum())},
    {'metric': 'Damaged share of eligible (%)', 'value': float(100 * eligible_pixels['damaged_gt10pct'].mean())},
    {'metric': 'Damaged distance p10 (km)', 'value': float(q_vals[0])},
    {'metric': 'Damaged distance p25 (km)', 'value': float(q_vals[1])},
    {'metric': 'Damaged distance p50 (km)', 'value': float(q_vals[2])},
    {'metric': 'Damaged distance p75 (km)', 'value': float(q_vals[3])},
    {'metric': 'Damaged distance p90 (km)', 'value': float(q_vals[4])},
    {'metric': 'Distance containing 50% of damage (km)', 'value': float(cum_distance(d_sorted, 0.50))},
    {'metric': 'Distance containing 80% of damage (km)', 'value': float(cum_distance(d_sorted, 0.80))},
    {'metric': 'Distance containing 90% of damage (km)', 'value': float(cum_distance(d_sorted, 0.90))},
    {'metric': 'HDI 50% lower bound (km)', 'value': float(hdi50[0])},
    {'metric': 'HDI 50% upper bound (km)', 'value': float(hdi50[1])},
    {'metric': 'HDI 80% lower bound (km)', 'value': float(hdi80[0])},
    {'metric': 'HDI 80% upper bound (km)', 'value': float(hdi80[1])},
    {'metric': 'Peak distance (max damaged count, km)', 'value': float(centers[peak_count_idx])},
    {'metric': 'Peak distance (max damage rate, km)', 'value': float(centers[peak_rate_idx])},
])

summary_distance['value'] = summary_distance['value'].astype(float).round(3)
display(summary_distance)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

axes[0].hist(dist_dmg, bins=60, color='#d73027', alpha=0.85)
axes[0].axvline(float(centers[peak_count_idx]), color='black', linestyle='--', linewidth=1.4, label='Peak count distance')
axes[0].set_title('Damaged Forest Pixels: Distance to Track')
axes[0].set_xlabel('Distance to track (km)')
axes[0].set_ylabel('Damaged pixel count')
axes[0].legend(loc='upper right', frameon=True)

y = np.arange(1, len(d_sorted) + 1) / len(d_sorted)
axes[1].plot(d_sorted, y, color='#08519c', linewidth=2)
axes[1].axhline(0.5, color='gray', linestyle=':', linewidth=1)
axes[1].axhline(0.8, color='gray', linestyle=':', linewidth=1)
axes[1].axvline(cum_distance(d_sorted, 0.50), color='#238b45', linestyle='--', linewidth=1.4, label='50% distance')
axes[1].axvline(cum_distance(d_sorted, 0.80), color='#756bb1', linestyle='--', linewidth=1.4, label='80% distance')
axes[1].set_title('Cumulative Concentration of Damaged Forest Pixels')
axes[1].set_xlabel('Distance to track (km)')
axes[1].set_ylabel('Cumulative share of damaged forest pixels')
axes[1].set_ylim(0, 1.02)
axes[1].legend(loc='lower right', frameon=True)

plt.tight_layout()
plt.show()


## Added at End: Damaged vs Non-damaged Forests Within 80%/90% Distance Thresholds

This mirrors the mangrove threshold check and asks: within the 80% and 90% damaged-distance zones, what share of forests are damaged vs non-damaged, and how are they distributed spatially (side/quadrant/wind zone).


In [ ]:
# Threshold-zone composition for forests (same style as mangrove follow-up)
# Percentages are based on eligible forest pixels used in this notebook.

from scipy.spatial import cKDTree

# Ensure distance column exists
if 'distance_to_track_km' not in eligible_pixels.columns:
    eligible_pixels['distance_to_track_km'] = eligible_pixels.geometry.distance(line_union) / 1000.0

# Damaged mask
if 'damaged_gt10pct' not in eligible_pixels.columns:
    eligible_pixels['damaged_gt10pct'] = eligible_pixels['rel_change'] < REL_DAMAGE_THRESHOLD

damaged_dist = eligible_pixels.loc[eligible_pixels['damaged_gt10pct'], 'distance_to_track_km'].to_numpy()
if len(damaged_dist) == 0:
    raise ValueError('No damaged (>10%) forest pixels found.')

d80 = float(np.quantile(damaged_dist, 0.80))
d90 = float(np.quantile(damaged_dist, 0.90))
thresholds = [('80% damaged distance', d80), ('90% damaged distance', d90)]

# Load track points for side/quadrant attribution (nearest track point)
pts_path = track_dir / 'AL132025_pts.shp'
track_pts_full = coerce_track_to_wgs84(gpd.read_file(pts_path)).to_crs(3448)
track_pts = track_pts_full[track_pts_full.intersects(track_aoi_geom)].copy() if 'track_aoi_geom' in globals() else track_pts_full.copy()
if len(track_pts) == 0:
    track_pts = track_pts_full.copy()

pts_xy = np.c_[track_pts.geometry.x.to_numpy(), track_pts.geometry.y.to_numpy()]
tree = cKDTree(pts_xy)

# Wind threshold unions
sw = track_windswath.copy()
if 'RADII' in sw.columns:
    sw['RADII'] = pd.to_numeric(sw['RADII'], errors='coerce')
else:
    sw['RADII'] = np.nan

u64 = unary_union(sw.loc[sw['RADII'] == 64, 'geometry'].tolist()) if np.any(sw['RADII'] == 64) else None
u50 = unary_union(sw.loc[sw['RADII'] == 50, 'geometry'].tolist()) if np.any(sw['RADII'] == 50) else None
u34 = unary_union(sw.loc[sw['RADII'] == 34, 'geometry'].tolist()) if np.any(sw['RADII'] == 34) else None

def in_geom(gdf, geom):
    if geom is None or getattr(geom, 'is_empty', False):
        return np.zeros(len(gdf), dtype=bool)
    return gdf.geometry.within(geom).to_numpy()

overall_rows = []
side_tables = {}
quadrant_tables = {}
wind_tables = {}

for label, thr in thresholds:
    sub = eligible_pixels[eligible_pixels['distance_to_track_km'] <= thr].copy()

    # Overall percentages within threshold
    pct_damaged = 100.0 * sub['damaged_gt10pct'].mean()
    pct_non = 100.0 - pct_damaged
    overall_rows.append({
        'threshold_label': label,
        'distance_km': round(thr, 3),
        'pct_damaged_within': round(pct_damaged, 2),
        'pct_non_damaged_within': round(pct_non, 2),
    })

    # Side/quadrant via nearest track point
    xy = np.c_[sub.geometry.x.to_numpy(), sub.geometry.y.to_numpy()]
    _, idx = tree.query(xy, k=1)
    near_xy = pts_xy[idx]
    dx = xy[:, 0] - near_xy[:, 0]
    dy = xy[:, 1] - near_xy[:, 1]

    sub['side'] = np.where(dx >= 0, 'East', 'West')
    sub['quadrant'] = np.select(
        [
            (dx >= 0) & (dy >= 0),
            (dx < 0) & (dy >= 0),
            (dx >= 0) & (dy < 0),
            (dx < 0) & (dy < 0),
        ],
        ['NE', 'NW', 'SE', 'SW'],
        default='Unknown'
    )

    # Wind zone
    in64 = in_geom(sub, u64)
    in50 = in_geom(sub, u50)
    in34 = in_geom(sub, u34)
    sub['wind_zone'] = np.where(in64, 'inside_64', np.where(in50, '50_to_64', np.where(in34, '34_to_50', 'outside_34')))

    side_tbl = (
        sub.groupby('side')['damaged_gt10pct']
        .mean()
        .mul(100)
        .round(2)
        .rename('pct_damaged')
        .reset_index()
    )
    side_tbl['pct_non_damaged'] = (100.0 - side_tbl['pct_damaged']).round(2)
    side_tables[label] = side_tbl.sort_values('side').reset_index(drop=True)

    quad_tbl = (
        sub.groupby('quadrant')['damaged_gt10pct']
        .mean()
        .mul(100)
        .round(2)
        .rename('pct_damaged')
        .reset_index()
    )
    quad_tbl['pct_non_damaged'] = (100.0 - quad_tbl['pct_damaged']).round(2)
    quadrant_tables[label] = quad_tbl.sort_values('quadrant').reset_index(drop=True)

    wind_tbl = (
        sub.groupby('wind_zone')['damaged_gt10pct']
        .mean()
        .mul(100)
        .round(2)
        .rename('pct_damaged')
        .reset_index()
    )
    wind_tbl['pct_non_damaged'] = (100.0 - wind_tbl['pct_damaged']).round(2)
    wind_tables[label] = wind_tbl.sort_values('wind_zone').reset_index(drop=True)

overall_df = pd.DataFrame(overall_rows)
display(overall_df)

for label, _ in thresholds:
    print(f"\n{label} side breakdown (%):")
    display(side_tables[label])
    print(f"{label} quadrant breakdown (%):")
    display(quadrant_tables[label])
    print(f"{label} wind-zone breakdown (%):")
    display(wind_tables[label])
